Препроцесинг даних для класичних моделей

In [2]:
import re

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

Завантаження даних

In [18]:
train_df = pd.read_csv("../data/train.csv")
test_df = pd.read_csv("../data/test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

Train shape: (10003, 3)
Test shape: (3080, 3)


,text,label,label_text
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival
3,Can I track my card while it is in the process...,11,card_arrival
4,"How do I know if I will get my card, or if it ...",11,card_arrival


Перевіряємо валідність колонок та дадасету

In [19]:
assert {"text", "label", "label_text"}.issubset(train_df.columns)
assert {"text", "label", "label_text"}.issubset(test_df.columns)

In [20]:
label_mapping_check = (
    train_df
    .groupby("label")["label_text"]
    .nunique()
)

label_mapping_check.value_counts()

label_text
1    77
Name: count, dtype: int64

Робимо мінімальний препроцесинг:
- lowercase
- нормалізацію апострофів
- видалення зайвих пробілів
- збереження важливих заперечень

In [21]:
def minimal_preprocess(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()

    # Нормалізація апострофів
    text = text.replace("’", "'").replace("`", "'")

    # Прибираємо переноси рядків і табуляції
    text = text.replace("\n", " ").replace("\r", " ").replace("\t", " ")

    # Нормалізуємо пробіли
    text = re.sub(r"\s+", " ", text)

    return text.strip()

Застосвуємо мінімальний препроцесинг. Щоб не змінювати оригінальний текст робимо нову колонку.

In [22]:
train_df['text_clean'] = train_df['text'].apply(minimal_preprocess)
test_df['text_clean'] = test_df['text'].apply(minimal_preprocess)

In [23]:
train_df[
    ["text", "text_clean", "label_text"]
].sample(10, random_state=42)

,text,text_clean,label_text
6883,Is it possible for me to change my PIN number?,is it possible for me to change my pin number?,change_pin
5836,I'm not sure why my card didn't work,i'm not sure why my card didn't work,declined_card_payment
8601,I don't think my top up worked,i don't think my top up worked,top_up_failed
2545,Can you explain why my payment was charged a fee?,can you explain why my payment was charged a fee?,card_payment_fee_charged
8697,How long does a transfer from a UK account tak...,how long does a transfer from a uk account tak...,balance_not_updated_after_bank_transfer
5573,Why am I getting declines when trying to make ...,why am i getting declines when trying to make ...,declined_transfer
576,What is the $1 transaction on my account?,what is the $1 transaction on my account?,extra_charge_on_statement
6832,It looks like my card payment was sent back.,it looks like my card payment was sent back.,reverted_card_payment?
7111,Why am I unable to transfer money when I was a...,why am i unable to transfer money when i was a...,beneficiary_not_allowed
439,What if there is an error on the exchange rate?,what if there is an error on the exchange rate?,card_payment_wrong_exchange_rate


Перевірка порожніх текстів

In [24]:
print(
    "Empty train texts:",
    train_df["text_clean"].eq("").sum()
)

print(
    "Empty test texts:",
    test_df["text_clean"].eq("").sum()
)

Empty train texts: 0
Empty test texts: 0


Перевірка дублікатів після очищення

In [25]:
train_duplicates = train_df["text_clean"].duplicated().sum()
test_duplicates = test_df["text_clean"].duplicated().sum()

print("Train duplicate texts:", train_duplicates)
print("Test duplicate texts:", test_duplicates)

Train duplicate texts: 4
Test duplicate texts: 1


In [26]:
duplicate_rows = train_df[
    train_df["text_clean"].duplicated(keep=False)
].sort_values("text_clean")

duplicate_rows[
    ["text", "text_clean", "label", "label_text"]
]

,text,text_clean,label,label_text
6910,Do I need to go to a physical bank to change m...,do i need to go to a physical bank to change m...,21,change_pin
6965,\nDo I need to go to a physical bank to change...,do i need to go to a physical bank to change m...,21,change_pin
1246,I can't seem to be able to use my card,i can't seem to be able to use my card,14,card_not_working
1290,\nI can't seem to be able to use my card\n\n\n,i can't seem to be able to use my card,14,card_not_working
1710,\nI put the wrong pin too many times and now i...,i put the wrong pin too many times and now it ...,49,pin_blocked
1724,I put the wrong pin too many times and now it ...,i put the wrong pin too many times and now it ...,49,pin_blocked
4594,Where can I withdraw money from?,where can i withdraw money from?,3,atm_support
4595,\nWhere can I withdraw money from?,where can i withdraw money from?,3,atm_support


Видаляємо дублікати

In [27]:
train_df = (
    train_df
    .drop_duplicates(subset='text_clean')
    .reset_index(drop=True)
)

In [28]:
print(train_df['text_clean'].duplicated().sum())

0


In [29]:
conflicting_duplicates = (
    train_df
    .groupby('text_clean')['label']
    .nunique()
)

conflicting_duplicates = conflicting_duplicates[
    conflicting_duplicates > 1
]

print(
    'Duplicate texts with different labels:',
    len(conflicting_duplicates)
)

Duplicate texts with different labels: 0


Розбиваємо train_df на train_part та validation_part. Test_df залишаємо недоторканим.

In [30]:
train_part, val_part = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df['label'],
)

print('Train part:', train_part.shape)
print('Validation part:', val_part.shape)
print('Test part:', test_df.shape)

Train part: (7999, 4)
Validation part: (2000, 4)
Test part: (3080, 4)


In [31]:
X_train = train_part['text_clean']
y_train = train_part['label']

X_val = val_part['text_clean']
y_val = val_part['label']

X_test = test_df['text_clean']
y_test = test_df['label']

Перевіряємо розподіл класів після split

In [32]:
split_distribution = pd.DataFrame({
    'full_train': train_df['label'].value_counts(normalize=True),
    'train': y_train.value_counts(normalize=True),
    'validation': y_val.value_counts(normalize=True),
}).fillna(0)

split_distribution.head()

,full_train,train,validation
label,,,
0,0.015902,0.015877,0.0160
1,0.011001,0.011001,0.0110
2,0.012601,0.012627,0.0125
3,0.008601,0.008626,0.0085
4,0.012701,0.012752,0.0125


Збереження підготовлених наборів

In [34]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

train_part.to_csv(
    "../data/processed/train_preprocessed.csv",
    index=False
)

val_part.to_csv(
    "../data/processed/validation_preprocessed.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test_preprocessed.csv",
    index=False
)

Для збереження потенційно інформативних токенів, таких як заперечення та вирази, пов'язані з проблемою, було застосовано мінімальну попередню обробку тексту. Конвеєр попередньої обробки включає використання нижніх літер, нормалізацію апострофа, нормалізацію пробілів та видалення порожніх значень. Агресивне видалення стоп-слів або лематизація не застосовувалися, оскільки повідомлення в датасеті короткі, а такі слова, як not, why, failed, та wrong можуть містити важливу інформацію про наміри.